In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# page 140
import pandas as pd

df = pd.read_csv("../data/ragas_synthetic_dataset.csv")
df.head(2)

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What is the significance of The White House's ...,"['1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국, 안전하...","The White House's recent executive order, sign...",AI Ethics Compliance Officer,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
1,G7이 첨단 AI 시스템을 위한 국제 행동강령(International Code o...,"['SPRi AI Brief |\n2023-12월호\nG7, 히로시마 AI 프로세스...",G7은 2023년 10월 30일 '히로시마 AI 프로세스'를 통해 AI 기업을 대상...,AI Policy Analyst,MISSPELLED,LONG,single_hop_specific_query_synthesizer


In [3]:
from datasets import Dataset

test_dataset = Dataset.from_pandas(df)
test_dataset

d:\skc0902\rag_two_branch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name'],
    num_rows: 10
})

In [4]:
# page 141

# "reference_contexts"열이 문자열이면, 리스트로 변환
import ast

def convert_to_list(example):
    value = example["reference_contexts"]
    if isinstance(value, str):                  # value 가 문자열인지 확인
        # print("isinstance")
        value = ast.literal_eval(value)         # 문자열을 리스트로 변환

    if not isinstance(value, list):             # value가 리스트가 아니면 오류 발생
        # print("not isinstance")
        raise TypeError(f"reference_contexts must be a list, got {type(value).__name__}")

    return {"reference_contexts": value}

test_dataset = test_dataset.map(convert_to_list)

# 변경된 값을 확인하려면
print(test_dataset["reference_contexts"])
print("------")
print(test_dataset[0]["reference_contexts"])


Map: 100%|██████████| 10/10 [00:00<00:00, 594.46 examples/s]

Column([['1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국, 안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령 발표\nKEY Contents\nn 미국 바이든 대통령이 ‘안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령’에 서명하고\n광범위한 행정 조치를 명시\nn 행정명령은 △AI의 안전과 보안 기준 마련 △개인정보보호 △형평성과 시민권 향상 △소비자\n보호 △노동자 지원 △혁신과 경쟁 촉진 △국제협력을 골자로 함\n£바이든 대통령, AI 행정명령 통해 안전하고 신뢰할 수 있는 AI 개발과 활용 추진\nn 미국 바이든 대통령이 2023년 10월 30일 연방정부 차원에서 안전하고 신뢰할 수 있는 AI 개발과\n사용을 보장하기 위한 행정명령을 발표\n∙ 행정명령은 △AI의 안전과 보안 기준 마련 △개인정보보호 △형평성과 시민권 향상 △소비자 보호\n△노동자 지원 △혁신과 경쟁 촉진 △국제협력에 관한 내용을 포괄\nn (AI 안전과 보안 기준) 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한\n주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및\nAI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진\n∙ △1026 플롭스(FLOPS, Floating Point Operation Per Second)를 초과하는 컴퓨팅 성능 또는 생물학적\n서열 데이터를 주로 사용하고 1023플롭스를 초과하는 컴퓨팅 성능을 사용하는 모델 △단일 데이터센터에서\n1,000Gbit/s 이상의 네트워킹으로 연결되며 AI 훈련에서 이론상 최대 1020 플롭스를 처리할 수 있는\n컴퓨팅 용량을 갖춘 컴퓨팅 클러스터가 정보공유 요구대상\nn (형평성과 시민권 향상) 법률, 주택, 보건 분야에서 AI의 무책임한 사용으로 인한 차별과 편견 및 기타\n문제를 방지하는 조치를 확대\n∙ 형사사법 시스템에서 AI 사용 모범사례를 개발하고, 주택 임대 시 AI 알고리즘 차별을 막

In [5]:
# 평가용 RAG 생성.

from langchain_text_splitters               import RecursiveCharacterTextSplitter
from langchain_community.document_loaders   import PyMuPDFLoader

from langchain_community.vectorstores       import FAISS
# from langchain_core.vectorstores            import InMemoryVectorStore

from langchain_core.output_parsers          import StrOutputParser
from langchain_core.runnables               import RunnablePassthrough
from langchain_core.prompts                 import PromptTemplate
from langchain_openai                       import ChatOpenAI, OpenAIEmbeddings

loader = PyMuPDFLoader("../data/SPRI_AI_Brief_2023년12월호_F.pdf") # 단계 1: 문서 로드(Load Documents)
docs = loader.load()

                                                                # 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)

embeddings = OpenAIEmbeddings()                         # 단계 3: 임베딩(Embedding) 생성

                                                        # 단계 4: DB 생성(Create DB) 및 저장
                                                        # 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)
# vectorstore = InMemoryVectorStore.from_documents(documents=split_documents,embedding=embeddings,)
                                                        # 단계 5: 검색기(Retriever) 생성
retriever = vectorstore.as_retriever()                  # 문서에 포함되어 있는 정보를 검색하고 생성합니다.

                                                        # 단계 6: 프롬프트 생성(Create Prompt)
prompt = PromptTemplate.from_template(                  # 프롬프트를 생성합니다.
    """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 

#Context: 
{context}

#Question:
{question}

#Answer:"""
)
                                                        # 단계 7: 언어모델(LLM) 생성
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)    # 모델(LLM) 을 생성합니다.

chain = (                                               # 단계 8: 체인(Chain) 생성
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [6]:
# page 142

batch_dataset = list(test_dataset["user_input"])
batch_dataset[:3]

["What is the significance of The White House's recent executive order on AI?",
 'G7이 첨단 AI 시스템을 위한 국제 행동강령(International Code of Conduct for Advanced AI Systems)을 어떻게 마련했는지에 대한 자세한 설명을 해줄 수 있나요?',
 '영국 AI 안전성 정상회의에서 뭐가 발표되었나요?']

In [7]:
# page 143

# 각 질문에 대해 RAG가 실제로 검색한 문서와 생성한 답변을 저장

# retrieved_documents : PDF 임베딩값.
retrieved_documents = retriever.batch(batch_dataset)
retrieved_contexts = [
    [doc.page_content for doc in documents]
    for documents in retrieved_documents
]

# LLM체인 최종 답변 저장
responses = chain.batch(batch_dataset)
responses

["The significance of The White House's recent executive order on AI, signed by President Biden on October 30, 2023, lies in its comprehensive approach to ensuring the safe and trustworthy development and use of AI. The executive order outlines several key areas of focus, including:\n\n1. Establishing safety and security standards for AI systems.\n2. Enhancing privacy protection.\n3. Promoting fairness and civil rights.\n4. Protecting consumers.\n5. Supporting workers.\n6. Encouraging innovation and competition.\n7. Fostering international cooperation.\n\nThe order requires companies developing powerful AI systems to share safety test results and key information with the U.S. government. It also aims to prevent discrimination and bias in areas like law, housing, and healthcare, and to promote responsible AI use in these sectors. Additionally, it supports AI research and infrastructure development across the U.S., and modernizes visa and interview processes to attract foreign experts in

In [8]:
# Ragas 0.4.x 평가에 필요한 RAG 실행 결과 열 추가

# "retrieved_contexts"열을 추가함.
for column_name, values in {
    "retrieved_contexts": retrieved_contexts,
    "response": responses,
}.items():
    if column_name in test_dataset.column_names:
        test_dataset = test_dataset.remove_columns(column_name)
    test_dataset = test_dataset.add_column(column_name, values)

print(test_dataset.column_names)
test_dataset.select_columns([
    "user_input", "retrieved_contexts", "response", "reference"
])

['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name', 'retrieved_contexts', 'response']


Dataset({
    features: ['user_input', 'retrieved_contexts', 'response', 'reference'],
    num_rows: 10
})

In [9]:
# Ragas 0.4.x 레거시 evaluate()와 LangChain 모델의 호환 설정
from ragas import RunConfig, evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    AnswerRelevancy,
    Faithfulness,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
)

evaluator_llm = LangchainLLMWrapper(
    ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
        timeout=60,
        max_retries=1,
    )
)
evaluator_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(
        model="text-embedding-3-small",
        timeout=60,
        max_retries=1,
    )
)

metrics = [
    Faithfulness(),
    AnswerRelevancy(strictness=1),
    LLMContextPrecisionWithReference(),
    LLMContextRecall(),
]

result = evaluate(
    dataset=test_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=RunConfig(
        timeout=60,
        max_retries=2,
        max_wait=10,
        max_workers=4,
    ),
    batch_size=2,
    raise_exceptions=True,
)
result

C:\Users\user\AppData\Local\Temp\ipykernel_7776\2722126842.py:5: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
C:\Users\user\AppData\Local\Temp\ipykernel_7776\2722126842.py:5: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\user\AppData\Local\Temp\ipykernel_7776\2722126842.py:5: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\user\AppData\Lo

{'faithfulness': 0.9113, 'answer_relevancy': 0.7529, 'llm_context_precision_with_reference': 0.8333, 'context_recall': 0.8667}

In [10]:
# 평가 결과 
result_df = result.to_pandas()
result_df.head()

,user_input,retrieved_contexts,reference_contexts,response,reference,persona_name,query_style,query_length,faithfulness,answer_relevancy,llm_context_precision_with_reference,context_recall
0,What is the significance of The White House's ...,[1. 정책/법제 \n2. 기업/산업 \n3. 기술/연구 \n 4. 인력/교육\n...,"[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국, 안전하고...",The significance of The White House's recent e...,"The White House's recent executive order, sign...",AI Ethics Compliance Officer,MISSPELLED,MEDIUM,1.0,0.893917,1.000000,1.0
1,G7이 첨단 AI 시스템을 위한 국제 행동강령(International Code o...,"[SPRi AI Brief | \n2023-12월호\n2\nG7, 히로시마 AI ...","[SPRi AI Brief |\n2023-12월호\nG7, 히로시마 AI 프로세스를...",G7은 첨단 AI 시스템을 위한 국제 행동강령을 마련하기 위해 '히로시마 AI 프로...,G7은 2023년 10월 30일 '히로시마 AI 프로세스'를 통해 AI 기업을 대상...,AI Policy Analyst,MISSPELLED,LONG,1.0,0.726601,0.916667,1.0
2,영국 AI 안전성 정상회의에서 뭐가 발표되었나요?,[1. 정책/법제 \n2. 기업/산업 \n3. 기술/연구 \n 4. 인력/교육\n...,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n영국 AI 안전...,"영국 AI 안전성 정상회의에서 ""블레츨리 선언""이 발표되었습니다. 이 선언은 AI ...",영국 AI 안전성 정상회의에 참가한 28개국들은 AI 안전 보장을 위한 협력 방안을...,AI Ethics Compliance Officer,POOR_GRAMMAR,MEDIUM,1.0,0.653033,1.000000,1.0
3,미드저니 저작권 소송 왜 기각됐나?,"[SPRi AI Brief | \n2023-12월호\n4\n미국 법원, 예술가들이...","[SPRi AI Brief |\n2023-12월호\n미국 법원, 예술가들이 생성 A...","미드저니 저작권 소송은 주로 두 가지 이유로 기각되었습니다. 첫째, 소송을 제기한 ...",미드저니에 대한 저작권 소송은 미국 캘리포니아 북부지방법원에서 기각됐다. 기각 이유...,AI Policy Advisor,POOR_GRAMMAR,SHORT,0.8,0.863758,1.000000,1.0
4,저작권청은 뭐하는 곳이야?,"[지적\n∙AI는 새로운 이미지를 생성할 때 다양한 예술가의 작품을 참조하므로, 생...",[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국 연방거래위...,"저작권청은 저작권과 관련된 법과 정책 이슈를 조사하고, 이를 통해 입법과 규제 조치...","저작권청은 생성 AI와 관련된 저작권법과 정책 이슈를 조사하고 있으며, 폭넓은 의견...",AI Policy Advisor,POOR_GRAMMAR,SHORT,1.0,0.463076,0.583333,1.0
